# Cold Head Status Model: comparison and export

This notebook consolidates the old training experiments found in this folder. It keeps the clustering experiments for comparison, but exports a supervised model compatible with the GUI.

GUI compatibility note: `plotter.py` passes one raw ADC waveform at a time to `model.predict(...)`, so the exported estimator must accept a vector of 128 raw samples. Clustering can still be compared on engineered features, but the final GUI model is trained on raw waveforms.


## What was found in the old notebooks

- `DBSCAN.ipynb`, `OPTICS.ipynb`, and `KMEANS.ipynb` repeat the same loading and feature-extraction blocks.
- `DBSCAN_features.ipynb` is another clustering variant, with labels derived from dataset order.
- `incremental_DBSCAN.ipynb` contains an experimental online-DBSCAN idea, but is not directly compatible with scikit-learn inference in the GUI.
- `inference_real_time.ipynb` tests loading a clustering model and assigning a new sample by distance.
- `Random_forest.ipynb` is the notebook that produced the current `random_forest_model.pkl` style used by the GUI.

The label convention used by the current GUI is preserved:

| Label | GUI status |
|---:|---|
| 0 | Maintenance |
| 1 | Normal |
| 2 | Compressor Down |


In [ ]:
from pathlib import Path
import json
import os
import pickle
import time
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.cluster import DBSCAN, KMeans, OPTICS
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    adjusted_rand_score,
    calinski_harabasz_score,
    classification_report,
    confusion_matrix,
    davies_bouldin_score,
    normalized_mutual_info_score,
    silhouette_score,
)
from sklearn.model_selection import LeaveOneGroupOut, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import sys

warnings.filterwarnings("ignore", category=UserWarning)


In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "training_notebooks" else Path.cwd().resolve()

# Set this to the folder that contains the large training .pkl files on the training machine.
# Leave it empty to use PROJECT_ROOT / "datasets".
EXTERNAL_DATASET_DIR = r""

DATASET_DIR = Path(EXTERNAL_DATASET_DIR).expanduser() if EXTERNAL_DATASET_DIR else PROJECT_ROOT / "datasets"
DATASET_DIR = DATASET_DIR.resolve()
OUTPUT_DIR = Path.cwd().resolve() / "trained_models"
OUTPUT_DIR.mkdir(exist_ok=True)

CHANNEL_INDEX = 1  # The old notebooks generally trained on ADC channel 1, i.e. arr[:, 1:2, :].
RANDOM_STATE = 42
DATASET_PATTERNS = ("*.pkl", "*.pklp")

# The full dataset is large. Use balanced sampling for fast, repeatable experiments.
# Set to None to use all available waveforms.
MAX_SAMPLES_PER_FILE = 25000
MAX_SAMPLES_PER_CLASS = 50000

# Use the candidate path first. After validating the notebook output, copy/rename it to PROJECT_ROOT / "random_forest_model.pkl".
KMEANS_MODEL_OUTPUT_PATH = OUTPUT_DIR / "cold_head_status_model_candidate.pkl"
RANDOM_FOREST_MODEL_OUTPUT_PATH = OUTPUT_DIR / "random_forest_model_candidate.pkl"
KMEANS_REPORT_OUTPUT_PATH = OUTPUT_DIR / "cold_head_status_model_report.json"
RANDOM_FOREST_REPORT_OUTPUT_PATH = OUTPUT_DIR / "random_forest_model_report.json"

STATUS_NAMES = {
    0: "Maintenance",
    1: "Normal",
    2: "Compressor Down",
}

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATASET_DIR:", DATASET_DIR)
print("DATASET_DIR exists:", DATASET_DIR.exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

MODEL_MODULE_PATH = PROJECT_ROOT / "cold_head_status_model.py"
if not MODEL_MODULE_PATH.exists():
    MODEL_MODULE_PATH.write_text(r'''
import numpy as np
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler


class KMeansHealthClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, n_clusters=3, random_state=42, n_init=20, anomaly_margin=0.5, anomaly_label=3):
        self.n_clusters = n_clusters
        self.random_state = random_state
        self.n_init = n_init
        self.anomaly_margin = anomaly_margin
        self.anomaly_label = anomaly_label

    def fit(self, X, y):
        features = extract_waveform_features(X)
        self.scaler_ = StandardScaler()
        scaled_features = self.scaler_.fit_transform(features)
        self.kmeans_ = KMeans(n_clusters=self.n_clusters, random_state=self.random_state, n_init=self.n_init)
        cluster_labels = self.kmeans_.fit_predict(scaled_features)
        self.cluster_to_label_ = self._map_clusters_to_labels(cluster_labels, np.asarray(y))
        distances = self.kmeans_.transform(scaled_features)
        min_distances = np.min(distances, axis=1)
        self.distance_threshold_ = float(np.max(min_distances) + self.anomaly_margin)
        self.classes_ = np.unique(y)
        return self

    def predict(self, X):
        cluster_labels, min_distances = self.predict_clusters(X)
        status_labels = np.array([self.cluster_to_label_.get(int(cluster), self.anomaly_label) for cluster in cluster_labels], dtype=int)
        if self.distance_threshold_ is not None:
            status_labels[min_distances > self.distance_threshold_] = self.anomaly_label
        return status_labels

    def predict_clusters(self, X):
        self._check_is_fitted()
        features = extract_waveform_features(X)
        scaled_features = self.scaler_.transform(features)
        cluster_labels = self.kmeans_.predict(scaled_features)
        distances = self.kmeans_.transform(scaled_features)
        min_distances = np.min(distances, axis=1)
        return cluster_labels, min_distances

    def transform(self, X):
        self._check_is_fitted()
        features = extract_waveform_features(X)
        scaled_features = self.scaler_.transform(features)
        return self.kmeans_.transform(scaled_features)

    def _map_clusters_to_labels(self, cluster_labels, y):
        mapping = {}
        for cluster in np.unique(cluster_labels):
            cluster_y = y[cluster_labels == cluster]
            labels, counts = np.unique(cluster_y, return_counts=True)
            mapping[int(cluster)] = int(labels[np.argmax(counts)])
        return mapping

    def _check_is_fitted(self):
        if not hasattr(self, "kmeans_") or not hasattr(self, "scaler_"):
            raise RuntimeError("KMeansHealthClassifier is not fitted yet.")


def extract_waveform_features(X):
    X = np.asarray(X, dtype=float)
    if X.ndim == 1:
        X = X.reshape(1, -1)
    if X.ndim == 3:
        X = X[:, 1, :]
    if X.ndim != 2:
        raise ValueError(f"Expected waveform array with 1, 2 or 3 dimensions, got {X.shape}.")
    mean = X.mean(axis=1)
    std = X.std(axis=1)
    maximum = X.max(axis=1)
    minimum = X.min(axis=1)
    peak_to_peak = maximum - minimum
    return np.column_stack([mean, std, maximum, minimum, peak_to_peak])
''', encoding="utf-8")
    print("Created missing model module:", MODEL_MODULE_PATH)

from cold_head_status_model import KMeansHealthClassifier, extract_waveform_features

print("OUTPUT_DIR:", OUTPUT_DIR)


In [ ]:
# Edit these rules when you add the original training files locally.
# Matching is case-insensitive and uses filename substrings.
FILENAME_LABEL_RULES = {
    "head_new": 1,
    "arcetri_head": 1,
    "dataset_no_comp": 2,
    "no_comp": 2,
    "compressor_down": 2,
    "head_30k_ore": 0,
    "dataset_1020": 0,
    "dataset_k_receiver": 0,
    "maintenance": 0,
}

def infer_label_from_filename(path: Path):
    name = path.stem.lower()
    for token, label in FILENAME_LABEL_RULES.items():
        if token.lower() in name:
            return label
    return None


## Dataset naming notes

`dataset_nasa.pklp` is intentionally treated as unlabeled until its operating state is confirmed. The `.pklp` extension may have been used to exclude it from older `*.pkl` loops; this notebook can still inspect it and use it for external validation, but it will not enter supervised training unless a label rule is added.


## Dataset diagnostics

Run the next cell on the training machine and paste its output back into the chat. It does not train anything; it only inspects the `.pkl` files so we can confirm shapes, labels, channels and waveform length.


In [ ]:
def describe_pickle_object(obj):
    description = {"python_type": type(obj).__name__}
    try:
        description["len"] = len(obj)
    except TypeError:
        description["len"] = None

    if isinstance(obj, dict):
        description["dict_keys"] = list(obj.keys())[:20]

    try:
        arr = np.asarray(obj)
        description["array_shape"] = tuple(arr.shape)
        description["array_dtype"] = str(arr.dtype)
        if arr.dtype != object and arr.size:
            finite = arr[np.isfinite(arr)]
            description["finite_count"] = int(finite.size)
            if finite.size:
                description["min"] = float(np.min(finite))
                description["max"] = float(np.max(finite))
                description["mean"] = float(np.mean(finite))
                description["std"] = float(np.std(finite))
    except Exception as exc:
        description["array_error"] = repr(exc)

    return description

def inspect_dataset_folder(dataset_dir: Path = DATASET_DIR, max_files: int = 30):
    dataset_dir = Path(dataset_dir)
    print("DATASET_DIR =", dataset_dir)
    print("exists =", dataset_dir.exists())
    if not dataset_dir.exists():
        print("Folder not found. Set EXTERNAL_DATASET_DIR in the configuration cell.")
        return pd.DataFrame()

    files = []
    for pattern in DATASET_PATTERNS:
        files.extend(dataset_dir.glob(pattern))
    files = sorted(set(files))
    print("pkl_files =", len(files))
    if len(files) > max_files:
        print(f"showing first {max_files} files only")
    rows = []
    for path in files[:max_files]:
        print("\n---", path.name, "---")
        print("size_MB =", round(path.stat().st_size / 1024 / 1024, 2))
        print("inferred_label =", infer_label_from_filename(path), STATUS_NAMES.get(infer_label_from_filename(path), "unlabeled"))
        try:
            with open(path, "rb") as f:
                obj = pickle.load(f)
            info = describe_pickle_object(obj)
            for key, value in info.items():
                print(f"{key} = {value}")

            arr = np.asarray(obj)
            if arr.ndim == 3:
                print("interpreted_as = (samples, channels, points)")
                print("samples =", arr.shape[0], "channels =", arr.shape[1], "points =", arr.shape[2])
                for channel in range(arr.shape[1]):
                    channel_data = arr[:, channel, :]
                    print(f"channel_{channel}: min={np.nanmin(channel_data):.4f}, max={np.nanmax(channel_data):.4f}, mean={np.nanmean(channel_data):.4f}, std={np.nanstd(channel_data):.4f}")
            elif arr.ndim == 2:
                print("interpreted_as = (samples, points)")
                print("samples =", arr.shape[0], "points =", arr.shape[1])

            rows.append({"file": path.name, "size_MB": path.stat().st_size / 1024 / 1024, **info})
        except Exception as exc:
            print("ERROR =", repr(exc))
            rows.append({"file": path.name, "error": repr(exc)})

    summary = pd.DataFrame(rows)
    print("\n=== summary ===")
    if not summary.empty:
        display(summary)
    return summary

dataset_diagnostics = inspect_dataset_folder()


In [ ]:
def load_pickle_waveforms(path: Path, channel_index: int = CHANNEL_INDEX) -> np.ndarray:
    with open(path, "rb") as f:
        data = pickle.load(f)

    arr = np.asarray(data, dtype=float)

    if arr.ndim == 3:
        if not 0 <= channel_index < arr.shape[1]:
            raise ValueError(f"{path.name}: channel_index={channel_index} is outside shape {arr.shape}")
        arr = arr[:, channel_index, :]
    elif arr.ndim == 2:
        pass
    else:
        raise ValueError(f"{path.name}: expected shape (n_samples, n_channels, n_points) or (n_samples, n_points), got {arr.shape}")

    return arr

def discover_pickle_datasets(dataset_dir: Path = DATASET_DIR) -> pd.DataFrame:
    rows = []
    files = []
    for pattern in DATASET_PATTERNS:
        files.extend(dataset_dir.glob(pattern))

    for path in sorted(set(files)):
        if "model" in path.name.lower():
            continue
        try:
            waveforms = load_pickle_waveforms(path)
            rows.append({
                "path": path,
                "file": path.name,
                "label": infer_label_from_filename(path),
                "n_samples": waveforms.shape[0],
                "n_points": waveforms.shape[1],
                "min": float(np.min(waveforms)),
                "max": float(np.max(waveforms)),
            })
        except Exception as exc:
            rows.append({"path": path, "file": path.name, "label": None, "error": repr(exc)})
    return pd.DataFrame(rows)

datasets = discover_pickle_datasets()
datasets


In [ ]:
def sample_rows(X, max_samples, random_state=RANDOM_STATE):
    if max_samples is None or len(X) <= max_samples:
        return X
    rng = np.random.default_rng(random_state)
    indices = rng.choice(len(X), size=max_samples, replace=False)
    return X[np.sort(indices)]

def build_raw_training_set(dataset_table: pd.DataFrame, require_labels: bool = True, max_samples_per_file=MAX_SAMPLES_PER_FILE, max_samples_per_class=MAX_SAMPLES_PER_CLASS):
    X_parts = []
    y_parts = []
    meta_rows = []

    for row in dataset_table.itertuples():
        label = getattr(row, "label", None)
        if require_labels and pd.isna(label):
            continue

        waveforms = load_pickle_waveforms(Path(row.path))
        waveforms = sample_rows(waveforms, max_samples_per_file)
        X_parts.append(waveforms)

        if not pd.isna(label):
            y_parts.append(np.full(waveforms.shape[0], int(label), dtype=int))

        meta_rows.extend({"file": row.file, "sample_index": i, "label": None if pd.isna(label) else int(label)} for i in range(waveforms.shape[0]))

    if not X_parts:
        return np.empty((0, 0)), np.array([], dtype=int), pd.DataFrame(meta_rows)

    X = np.vstack(X_parts)
    y = np.concatenate(y_parts) if y_parts else np.array([], dtype=int)
    meta = pd.DataFrame(meta_rows)

    if require_labels and max_samples_per_class is not None and len(y):
        rng = np.random.default_rng(RANDOM_STATE)
        selected = []
        for label in np.unique(y):
            label_indices = np.flatnonzero(y == label)
            if len(label_indices) > max_samples_per_class:
                label_indices = rng.choice(label_indices, size=max_samples_per_class, replace=False)
            selected.extend(label_indices.tolist())
        selected = np.array(sorted(selected))
        X = X[selected]
        y = y[selected]
        meta = meta.iloc[selected].reset_index(drop=True)
    return X, y, meta

X_raw, y, meta = build_raw_training_set(datasets, require_labels=True)
print("X_raw:", X_raw.shape)
print("y:", y.shape, dict(zip(*np.unique(y, return_counts=True))) if len(y) else "no labeled data")


In [ ]:
def waveform_features(X: np.ndarray) -> pd.DataFrame:
    X = np.asarray(X, dtype=float)
    mean = X.mean(axis=1)
    std = X.std(axis=1)
    maximum = X.max(axis=1)
    minimum = X.min(axis=1)
    vpp = maximum - minimum
    rms = np.sqrt(np.mean(np.square(X), axis=1))

    centered = X - mean[:, None]
    safe_std = np.where(std == 0, 1.0, std)
    skew = np.mean((centered / safe_std[:, None]) ** 3, axis=1)
    kurtosis = np.mean((centered / safe_std[:, None]) ** 4, axis=1)

    return pd.DataFrame({
        "mean": mean,
        "std": std,
        "max": maximum,
        "min": minimum,
        "vpp": vpp,
        "rms": rms,
        "skew": skew,
        "kurtosis": kurtosis,
    })

# For clustering we can also use unlabeled datasets.
X_all, _, meta_all = build_raw_training_set(datasets, require_labels=False)
feature_table = waveform_features(X_all) if len(X_all) else pd.DataFrame()
feature_table.head()


In [ ]:
def safe_cluster_metrics(X, labels, y_true=None):
    result = {"n_clusters": len(set(labels) - {-1}), "noise": int(np.sum(labels == -1))}
    non_noise = labels != -1
    unique_non_noise = set(labels[non_noise])

    if np.sum(non_noise) > 2 and len(unique_non_noise) >= 2:
        result["silhouette"] = float(silhouette_score(X[non_noise], labels[non_noise]))
        result["calinski_harabasz"] = float(calinski_harabasz_score(X[non_noise], labels[non_noise]))
        result["davies_bouldin"] = float(davies_bouldin_score(X[non_noise], labels[non_noise]))
    else:
        result["silhouette"] = np.nan
        result["calinski_harabasz"] = np.nan
        result["davies_bouldin"] = np.nan

    if y_true is not None and len(y_true) == len(labels):
        y_true = np.asarray(y_true)
        known_mask = pd.notna(y_true)
        if np.sum(known_mask) > 2 and len(set(y_true[known_mask])) > 1:
            result["adjusted_rand"] = float(adjusted_rand_score(y_true[known_mask], labels[known_mask]))
            result["normalized_mutual_info"] = float(normalized_mutual_info_score(y_true[known_mask], labels[known_mask]))

    return result

def compare_clustering_methods(features: pd.DataFrame, y_true=None):
    if features.empty:
        return pd.DataFrame(), {}

    X_scaled = StandardScaler().fit_transform(features)
    pca = PCA(n_components=min(2, X_scaled.shape[1]), random_state=RANDOM_STATE)
    X_cluster = pca.fit_transform(X_scaled)

    methods = {
        "KMeans-3": KMeans(n_clusters=3, n_init=20, random_state=RANDOM_STATE),
        "DBSCAN": DBSCAN(eps=0.8, min_samples=5),
        "OPTICS": OPTICS(min_samples=5, xi=0.05, min_cluster_size=0.05),
    }

    rows = []
    fitted = {}
    for name, estimator in methods.items():
        start = time.perf_counter()
        labels = estimator.fit_predict(X_cluster)
        elapsed = time.perf_counter() - start
        row = {"method": name, "fit_seconds": elapsed}
        row.update(safe_cluster_metrics(X_cluster, labels, y_true=y_true))
        rows.append(row)
        fitted[name] = {"estimator": estimator, "labels": labels, "X_cluster": X_cluster, "pca": pca}

    results = pd.DataFrame(rows)
    sort_columns = [column for column in ["silhouette", "adjusted_rand"] if column in results.columns]
    if sort_columns:
        results = results.sort_values(sort_columns, ascending=False, na_position="last")
    return results, fitted

y_for_all = meta_all["label"].to_numpy() if "label" in meta_all else None
cluster_results, fitted_clusters = compare_clustering_methods(feature_table, y_true=y_for_all)
cluster_results


In [ ]:
if not feature_table.empty:
    best_name = cluster_results.iloc[0]["method"] if not cluster_results.empty else next(iter(fitted_clusters))
    best = fitted_clusters[best_name]
    plot_df = pd.DataFrame({
        "PC1": best["X_cluster"][:, 0],
        "PC2": best["X_cluster"][:, 1] if best["X_cluster"].shape[1] > 1 else 0,
        "cluster": best["labels"].astype(str),
        "file": meta_all["file"].to_numpy(),
    })

    fig, ax = plt.subplots(figsize=(8, 5))
    for cluster, group in plot_df.groupby("cluster"):
        ax.scatter(group["PC1"], group["PC2"], s=36, label=cluster, alpha=0.8)
    ax.set_title(f"Best clustering view: {best_name}")
    ax.set_xlabel("PCA component 1")
    ax.set_ylabel("PCA component 2")
    ax.legend(title="Cluster")
    ax.grid(alpha=0.25)
    plt.show()
else:
    print("No local feature data available for clustering.")


In [ ]:
def can_train_supervised_model(X, y):
    if len(X) == 0 or len(y) == 0:
        return False, "No labeled datasets were found. Add files whose names match FILENAME_LABEL_RULES."
    unique, counts = np.unique(y, return_counts=True)
    if len(unique) < 2:
        return False, f"Need at least 2 classes, found {dict(zip(unique, counts))}."
    if np.min(counts) < 2:
        return False, f"Each class needs at least 2 samples for a stratified split, found {dict(zip(unique, counts))}."
    return True, "OK"


## Paper-aligned deployable model: K-Means + feature extraction

The paper describes five features per 128-sample waveform: mean, standard deviation, maximum, minimum and peak-to-peak range. The deployable model below keeps the GUI interface unchanged: it accepts raw 128-sample waveforms, extracts those features internally, assigns the nearest K-Means cluster, maps clusters to health labels by majority vote, and marks samples beyond the centroid-distance threshold as `Unknown Anomaly`.


In [ ]:
ready, reason = can_train_supervised_model(X_raw, y)
print(reason)

if ready:
    kmeans_health_model = KMeansHealthClassifier(
        n_clusters=3,
        random_state=RANDOM_STATE,
        n_init=20,
        anomaly_margin=0.5,
        anomaly_label=3,
    )
    kmeans_health_model.fit(X_raw, y)

    y_pred = kmeans_health_model.predict(X_raw)
    known_mask = y_pred != 3
    ordered_labels = sorted(np.unique(y))

    kmeans_report = {
        "model_type": "KMeansHealthClassifier",
        "feature_names": ["mean", "std", "max", "min", "peak_to_peak"],
        "labels": STATUS_NAMES,
        "cluster_to_label": kmeans_health_model.cluster_to_label_,
        "distance_threshold": kmeans_health_model.distance_threshold_,
        "anomaly_margin": kmeans_health_model.anomaly_margin,
        "channel_index": CHANNEL_INDEX,
        "n_raw_points": int(X_raw.shape[1]),
        "training_files": datasets[["file", "label", "n_samples", "n_points"]].to_dict(orient="records"),
        "training_accuracy_known_labels": float(accuracy_score(y[known_mask], y_pred[known_mask])) if np.any(known_mask) else None,
        "unknown_anomaly_count_on_training": int(np.sum(~known_mask)),
        "classification_report_known_labels": classification_report(
            y[known_mask],
            y_pred[known_mask],
            labels=ordered_labels,
            target_names=[STATUS_NAMES[i] for i in ordered_labels],
            output_dict=True,
        ) if np.any(known_mask) else None,
        "confusion_matrix_known_labels": confusion_matrix(y[known_mask], y_pred[known_mask], labels=ordered_labels).tolist() if np.any(known_mask) else None,
    }

    joblib.dump(kmeans_health_model, KMEANS_MODEL_OUTPUT_PATH)
    KMEANS_REPORT_OUTPUT_PATH.write_text(json.dumps(kmeans_report, indent=2), encoding="utf-8")

    print(f"Saved paper-aligned GUI model to: {KMEANS_MODEL_OUTPUT_PATH}")
    print(f"Saved report to: {KMEANS_REPORT_OUTPUT_PATH}")
    print("Cluster to status label:", {cluster: STATUS_NAMES.get(label, label) for cluster, label in kmeans_health_model.cluster_to_label_.items()})
    print("Distance threshold:", kmeans_health_model.distance_threshold_)
    print("Unknown anomalies on training set:", kmeans_report["unknown_anomaly_count_on_training"])
    print(classification_report(y[known_mask], y_pred[known_mask], labels=ordered_labels, target_names=[STATUS_NAMES[i] for i in ordered_labels]))
else:
    print("K-Means export skipped.")


## Supervised baseline: Random Forest on raw waveforms

This keeps the old GUI model style as a comparison baseline. The paper-aligned K-Means model above is the preferred deployable model.


In [ ]:
ready, reason = can_train_supervised_model(X_raw, y)
print(reason)

if ready:
    X_train, X_test, y_train, y_test = train_test_split(
        X_raw, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
    )

    gui_model = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", RandomForestClassifier(
            n_estimators=300,
            random_state=RANDOM_STATE,
            class_weight="balanced",
            n_jobs=-1,
        )),
    ])
    gui_model.fit(X_train, y_train)

    y_pred = gui_model.predict(X_test)
    ordered_labels = sorted(np.unique(y))
    report = {
        "accuracy": float(accuracy_score(y_test, y_pred)),
        "labels": STATUS_NAMES,
        "classification_report": classification_report(y_test, y_pred, labels=ordered_labels, target_names=[STATUS_NAMES[i] for i in ordered_labels], output_dict=True),
        "confusion_matrix": confusion_matrix(y_test, y_pred, labels=ordered_labels).tolist(),
        "channel_index": CHANNEL_INDEX,
        "n_features": int(X_raw.shape[1]),
        "training_files": datasets[["file", "label", "n_samples", "n_points"]].to_dict(orient="records"),
    }

    joblib.dump(gui_model, RANDOM_FOREST_MODEL_OUTPUT_PATH)
    RANDOM_FOREST_REPORT_OUTPUT_PATH.write_text(json.dumps(report, indent=2), encoding="utf-8")

    print(f"Saved Random Forest baseline to: {RANDOM_FOREST_MODEL_OUTPUT_PATH}")
    print(f"Saved report to: {RANDOM_FOREST_REPORT_OUTPUT_PATH}")
    print("Accuracy:", report["accuracy"])
    print(classification_report(y_test, y_pred, labels=ordered_labels, target_names=[STATUS_NAMES[i] for i in ordered_labels]))
else:
    print("Supervised export skipped.")


## Leave-one-file-out validation

This validation is stricter than a random train/test split: one acquisition file is held out entirely, so the model must generalize to a different run/head. The compressor-down class has only one source file, so that fold is skipped because the training set would not contain all classes.


In [ ]:
def leave_one_file_out_random_forest(X, y, meta):
    groups = meta["file"].to_numpy()
    ordered_labels = sorted(np.unique(y))
    rows = []
    skipped = []

    for train_index, test_index in LeaveOneGroupOut().split(X, y, groups):
        held_out_file = groups[test_index][0]
        train_labels = set(y[train_index])
        test_labels = set(y[test_index])
        if set(ordered_labels) - train_labels:
            skipped.append({
                "held_out_file": held_out_file,
                "reason": f"training would miss labels {sorted(set(ordered_labels) - train_labels)}",
            })
            continue

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", RandomForestClassifier(
                n_estimators=300,
                random_state=RANDOM_STATE,
                class_weight="balanced",
                n_jobs=-1,
            )),
        ])
        model.fit(X[train_index], y[train_index])
        pred = model.predict(X[test_index])

        row = {
            "held_out_file": held_out_file,
            "true_status": STATUS_NAMES[int(y[test_index][0])],
            "n_samples": int(len(test_index)),
            "accuracy": float(accuracy_score(y[test_index], pred)),
        }
        for value, count in zip(*np.unique(pred, return_counts=True)):
            row[STATUS_NAMES.get(int(value), "Unknown Anomaly")] = int(count)
        rows.append(row)

    return pd.DataFrame(rows).fillna(0), pd.DataFrame(skipped)

if len(X_raw) and len(y):
    loo_results, loo_skipped = leave_one_file_out_random_forest(X_raw, y, meta)
    display(loo_results)
    if not loo_skipped.empty:
        print("Skipped folds:")
        display(loo_skipped)
else:
    print("No labeled data available.")


## Per-file model diagnostics

These tables show whether a model is learning the physical classes or mostly separating individual dataset files. This is especially important because random sample splits can be optimistic when adjacent waveform windows come from the same acquisition run.


In [ ]:
def summarize_predictions_by_file(model, X, meta, true_labels=None, status_names=STATUS_NAMES):
    predictions = model.predict(X)
    summary = meta.copy().reset_index(drop=True)
    summary["predicted_label"] = predictions
    summary["predicted_status"] = summary["predicted_label"].map(status_names).fillna("Unknown Anomaly")
    if true_labels is not None and len(true_labels) == len(summary):
        summary["true_label"] = true_labels
        summary["true_status"] = summary["true_label"].map(status_names)
        summary["correct"] = summary["true_label"] == summary["predicted_label"]

    counts = (
        summary.groupby(["file", "predicted_status"])
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )

    if "correct" in summary:
        accuracy_by_file = summary.groupby("file")["correct"].mean().rename("accuracy").reset_index()
        counts = counts.merge(accuracy_by_file, on="file", how="left")

    return counts, summary

if "kmeans_health_model" in globals():
    kmeans_file_counts, kmeans_predictions = summarize_predictions_by_file(kmeans_health_model, X_raw, meta, y)
    display(kmeans_file_counts)

    print("K-Means confusion matrix on sampled training data:")
    ordered_labels = sorted(np.unique(y))
    print(confusion_matrix(y, kmeans_health_model.predict(X_raw), labels=ordered_labels))

if "gui_model" in globals():
    rf_file_counts, rf_predictions = summarize_predictions_by_file(gui_model, X_raw, meta, y)
    display(rf_file_counts)


## Unlabeled external dataset check

If `dataset_nasa.pklp` or other unlabeled files are present, run the trained models on them without using them for training. A high anomaly rate or a dominant unexpected class is a useful clue.


In [ ]:
unlabeled_rows = datasets[datasets["label"].isna()] if "label" in datasets else pd.DataFrame()
for row in unlabeled_rows.itertuples():
    X_external = sample_rows(load_pickle_waveforms(Path(row.path)), MAX_SAMPLES_PER_FILE)
    print()
    print("External dataset:", row.file, X_external.shape)
    if "kmeans_health_model" in globals():
        pred = kmeans_health_model.predict(X_external)
        values, counts = np.unique(pred, return_counts=True)
        print("K-Means:", {STATUS_NAMES.get(int(v), "Unknown Anomaly"): int(c) for v, c in zip(values, counts)})
        _, distances = kmeans_health_model.predict_clusters(X_external)
        print("K-Means min distance: min", float(np.min(distances)), "median", float(np.median(distances)), "max", float(np.max(distances)))
    if "gui_model" in globals():
        pred = gui_model.predict(X_external)
        values, counts = np.unique(pred, return_counts=True)
        print("Random Forest:", {STATUS_NAMES.get(int(v), "Unknown Anomaly"): int(c) for v, c in zip(values, counts)})


In [ ]:
# Smoke test for the exported model. This mirrors what plotter.py does in the GUI.
if KMEANS_MODEL_OUTPUT_PATH.exists() and len(X_raw):
    loaded_model = joblib.load(KMEANS_MODEL_OUTPUT_PATH)
    sample = X_raw[0].reshape(1, -1)
    predicted_label = int(loaded_model.predict(sample)[0])
    print("Predicted label:", predicted_label, STATUS_NAMES.get(predicted_label, "Unknown"))
else:
    print("No exported model found yet.")


## Report figures

The following cells generate compact figures suitable for the technical report. Recommended attachments:

1. Dataset composition by file and class.
2. PCA projection of the engineered features.
3. Clustering comparison metrics.
4. K-Means confusion matrix, to document why it was not selected for the GUI.
5. Random Forest confusion matrix and leave-one-file-out validation.
6. External `dataset_nasa.pklp` prediction summary.
7. Representative raw waveform examples for each operational state.

All figures are saved under `trained_models/report_figures/`.


In [ ]:
FIGURE_DIR = OUTPUT_DIR / "report_figures"
FIGURE_DIR.mkdir(exist_ok=True)

REPORT_COLORS = {
    "Maintenance": "#f39c12",
    "Normal": "#27ae60",
    "Compressor Down": "#e74c3c",
    "Unknown Anomaly": "#9b59b6",
    "Unlabeled": "#7f8c8d",
}

plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 180,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

def save_current_figure(name):
    path = FIGURE_DIR / name
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    print("saved", path)
    plt.show()

# 1. Labelled dataset composition (external/unlabelled files are excluded).
if not datasets.empty:
    composition = datasets[datasets["label"].notna()].copy()
    composition["status"] = composition["label"].map(STATUS_NAMES)
    composition = composition.sort_values(["status", "file"])

    fig, ax = plt.subplots(figsize=(10, 4.8))
    colors = [REPORT_COLORS.get(status, "#34495e") for status in composition["status"]]
    ax.barh(composition["file"], composition["n_samples"], color=colors)
    ax.set_title("Labelled dataset composition")
    ax.set_xlabel("Samples per selected channel")
    ax.set_ylabel("")
    for i, row in enumerate(composition.itertuples()):
        ax.text(row.n_samples, i, f"  {row.status} ({int(row.n_samples):,})", va="center", fontsize=8)
    save_current_figure("01_dataset_composition.png")

# 1b. Pointwise means of selected raw waveform windows by class.
if len(X_raw) and len(y):
    rng = np.random.default_rng(RANDOM_STATE)
    ordered_labels = sorted(np.unique(y))
    fig, axes = plt.subplots(len(ordered_labels), 1, figsize=(10, 7.2), sharex=True, sharey=True)
    if len(ordered_labels) == 1:
        axes = [axes]

    n_examples_per_class = 12
    sample_axis = np.arange(X_raw.shape[1])

    for ax, label in zip(axes, ordered_labels):
        label_indices = np.flatnonzero(y == label)
        selected = rng.choice(
            label_indices,
            size=min(n_examples_per_class, len(label_indices)),
            replace=False,
        )
        status = STATUS_NAMES[int(label)]
        color = REPORT_COLORS.get(status, "#34495e")
        waveforms = X_raw[selected]

        ax.plot(
            sample_axis,
            waveforms.mean(axis=0),
            color=color,
            linewidth=2.6,
            label=f"Pointwise mean (n={len(waveforms)})",
        )
        ax.set_title(status)
        ax.set_ylabel("ADC value")
        ax.legend(loc="upper right")

    axes[-1].set_xlabel("Sample index")
    fig.suptitle(
        "Pointwise means of 12 recorded 128-sample windows by operating category\n"
        f"Second acquisition channel; random seed {RANDOM_STATE}",
        y=1.01,
    )
    save_current_figure("08_raw_waveform_examples_by_status.png")

# 2. PCA projection of engineered features.
if not feature_table.empty:
    X_features_scaled = StandardScaler().fit_transform(feature_table)
    pca_2d = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X_features_scaled)
    pca_plot = meta_all.copy().reset_index(drop=True)
    pca_plot["PC1"] = pca_2d[:, 0]
    pca_plot["PC2"] = pca_2d[:, 1]
    pca_plot["status"] = pca_plot["label"].map(STATUS_NAMES).fillna("Unlabeled")

    fig, ax = plt.subplots(figsize=(8, 6))
    for status, group in pca_plot.groupby("status"):
        ax.scatter(
            group["PC1"],
            group["PC2"],
            s=7,
            alpha=0.45,
            label=status,
            color=REPORT_COLORS.get(status, "#34495e"),
        )
    ax.set_title("PCA projection of waveform statistical features")
    ax.set_xlabel("PCA component 1")
    ax.set_ylabel("PCA component 2")
    ax.legend(markerscale=2)
    save_current_figure("02_feature_pca_by_status.png")

# 3. Clustering comparison metrics.
if "cluster_results" in globals() and not cluster_results.empty:
    metrics = [col for col in ["silhouette", "adjusted_rand", "normalized_mutual_info"] if col in cluster_results.columns]
    if metrics:
        plot_df = cluster_results.set_index("method")[metrics]
        fig, ax = plt.subplots(figsize=(8, 4.8))
        plot_df.plot(kind="bar", ax=ax)
        ax.set_title("Clustering method comparison")
        ax.set_ylabel("Score")
        ax.set_xlabel("")
        ax.tick_params(axis="x", rotation=0)
        ax.legend(loc="best")
        save_current_figure("03_clustering_metrics.png")

# 4. K-Means confusion matrix.
if "kmeans_health_model" in globals() and len(y):
    ordered_labels = sorted(np.unique(y))
    y_pred_kmeans = kmeans_health_model.predict(X_raw)
    cm = confusion_matrix(y, y_pred_kmeans, labels=ordered_labels)
    fig, ax = plt.subplots(figsize=(5.8, 5.2))
    image = ax.imshow(cm, cmap="Oranges")
    ax.set_title("K-Means confusion matrix")
    ax.set_xticks(range(len(ordered_labels)), [STATUS_NAMES[i] for i in ordered_labels], rotation=25, ha="right")
    ax.set_yticks(range(len(ordered_labels)), [STATUS_NAMES[i] for i in ordered_labels])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center", color="black")
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    save_current_figure("04_kmeans_confusion_matrix.png")

# 5. Random Forest confusion matrix on the random holdout split if y_test/y_pred are still in memory.
if "gui_model" in globals() and len(y):
    ordered_labels = sorted(np.unique(y))
    rf_pred_all = gui_model.predict(X_raw)
    cm = confusion_matrix(y, rf_pred_all, labels=ordered_labels)
    fig, ax = plt.subplots(figsize=(5.8, 5.2))
    image = ax.imshow(cm, cmap="Greens")
    ax.set_title("Random Forest confusion matrix on sampled data")
    ax.set_xticks(range(len(ordered_labels)), [STATUS_NAMES[i] for i in ordered_labels], rotation=25, ha="right")
    ax.set_yticks(range(len(ordered_labels)), [STATUS_NAMES[i] for i in ordered_labels])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center", color="black")
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    save_current_figure("05_random_forest_confusion_matrix.png")

# 6. Leave-one-file-out validation.
if "loo_results" in globals() and not loo_results.empty:
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ordered = loo_results.sort_values("accuracy")
    ax.barh(ordered["held_out_file"], ordered["accuracy"], color="#3daee9")
    ax.set_title("Random Forest leave-one-file-out validation")
    ax.set_xlabel("Accuracy")
    ax.set_xlim(0, 1.02)
    for i, row in enumerate(ordered.itertuples()):
        ax.text(row.accuracy, i, f"  {row.accuracy:.3f}", va="center")
    save_current_figure("06_leave_one_file_out_accuracy.png")

# 7. External unlabeled dataset predictions.
external_summary_rows = []
unlabeled_rows = datasets[datasets["label"].isna()] if "label" in datasets else pd.DataFrame()
for row in unlabeled_rows.itertuples():
    X_external = sample_rows(load_pickle_waveforms(Path(row.path)), MAX_SAMPLES_PER_FILE)
    if "kmeans_health_model" in globals():
        values, counts = np.unique(kmeans_health_model.predict(X_external), return_counts=True)
        for value, count in zip(values, counts):
            external_summary_rows.append({"file": row.file, "model": "K-Means", "status": STATUS_NAMES.get(int(value), "Unknown Anomaly"), "count": int(count)})
    if "gui_model" in globals():
        values, counts = np.unique(gui_model.predict(X_external), return_counts=True)
        for value, count in zip(values, counts):
            external_summary_rows.append({"file": row.file, "model": "Random Forest", "status": STATUS_NAMES.get(int(value), "Unknown Anomaly"), "count": int(count)})

external_summary = pd.DataFrame(external_summary_rows)
if not external_summary.empty:
    fig, ax = plt.subplots(figsize=(8, 4.8))
    pivot = external_summary.pivot_table(index="model", columns="status", values="count", aggfunc="sum", fill_value=0)
    pivot.plot(kind="bar", stacked=True, ax=ax, color=[REPORT_COLORS.get(col, "#34495e") for col in pivot.columns])
    ax.set_title("Predictions on unlabeled external dataset")
    ax.set_xlabel("")
    ax.set_ylabel("Samples")
    ax.tick_params(axis="x", rotation=0)
    ax.legend(title="Predicted status")
    save_current_figure("07_external_dataset_predictions.png")

print("Figure directory:", FIGURE_DIR)


## Next steps

1. Add the original training `.pkl` files to `datasets/` or update `DATASET_DIR`.
2. Check `FILENAME_LABEL_RULES` so each dataset maps to the intended GUI class.
3. Run all cells and inspect clustering metrics, PCA plot, confusion matrix and classification report.
4. When satisfied, copy or rename `training_notebooks/trained_models/cold_head_status_model_candidate.pkl` to `cold_head_status_model.pkl` in the project root. The GUI will prefer this model and fall back to `random_forest_model.pkl` if it is absent.
5. Relaunch the GUI and verify live predictions against known cold-head states.
